# IndoML Datathon — Track 1: Noise Event Detection (Vaani)

**Frozen BEATs + FDY-CRNN, mean-teacher semi-supervision on a per-tier masked loss,
frequency MixStyle for domain shift, cSEBBs post-processing.**

Run top to bottom. Set the runtime to **GPU** first:
`Runtime -> Change runtime type -> T4 GPU`.

| Step | Cell | Notes |
|---|---|---|
| 0 | GPU check | fails loudly if you forgot to switch runtime |
| 1 | Clone + install | never installs torch (Colab's is CUDA-linked) |
| 2 | Prepare data | downloads Vaani once to Drive, cached locally for fast reads |
| 3 | BEATs weights | ~360 MB, cached to Drive if mounted |
| 4 | Train | reads local disk, checkpoints to Drive if mounted |
| 5 | Tune cSEBBs | no retraining, biggest ROI per minute |
| 6 | Predict | writes `submission.zip` (`predictions.jsonl`) |
| 7 | Validate | checks the archive against the Codabench format |


## 0 · GPU check


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> T4 GPU'
print('torch', torch.__version__, '| cuda', torch.version.cuda,
      '|', torch.cuda.get_device_name(0))


## 1 · Clone the repo and install dependencies

`requirements.txt` deliberately excludes torch: Colab ships a CUDA-linked build and
pip-installing torch here would replace it with a CPU wheel and silently kill the GPU.


In [ ]:
REPO = 'https://github.com/raut7218/vaani-sed-track1.git'
import os, sys
if not os.path.exists('/content/vaani-sed-track1'):
    !git clone -q $REPO /content/vaani-sed-track1
%cd /content/vaani-sed-track1
!git pull -q || true

# everything except torch/torchaudio, which Colab already has
!pip install -q soundfile librosa 'datasets>=2.18' huggingface_hub pyyaml tqdm
sys.path.insert(0, '/content/vaani-sed-track1')
print('ready')


### Mount Drive so the dataset and checkpoints survive a disconnect


In [ ]:
USE_DRIVE = True  # keep True: the dataset then downloads once and is reused

import os, subprocess, time

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/vaani_track1'
else:
    BASE = '/content/work'
os.makedirs(BASE, exist_ok=True)

# Training always reads from *local* disk, mounted or not. Drive is mounted over
# FUSE, and a shuffled DataLoader doing tens of thousands of small random reads
# against it is exactly what causes the "nothing happens for 20 minutes" stall
# before the first epoch prints anything. Drive only ever sees big sequential
# rsync transfers below, never per-sample reads during training.
DATA = '/content/work/data'          # local: training reads from here
DATA_DRIVE = f'{BASE}/data'          # persistent copy: Drive if mounted, else unused
RUN = f'{BASE}/runs/baseline'        # checkpoints: small, infrequent writes -> Drive is fine
os.makedirs(DATA, exist_ok=True)
os.makedirs(RUN, exist_ok=True)

if USE_DRIVE and os.path.exists(f'{DATA_DRIVE}/manifest.jsonl'):
    print('[data] cached dataset found on Drive - copying to local disk once ...')
    t0 = time.time()
    subprocess.run(['rsync', '-a', '--info=progress2', f'{DATA_DRIVE}/', f'{DATA}/'], check=True)
    print('[data] synced from %s in %.1fs' % (DATA_DRIVE, time.time() - t0))
else:
    print('[data] no cached dataset on Drive yet - the download cells below will fetch it')

print('DATA       =', DATA, ' (local, training reads from here)')
print('DATA_DRIVE =', DATA_DRIVE if USE_DRIVE else '(Drive not mounted - nothing persists)')
print('RUN        =', RUN)


## 2 · Hugging Face login (the dataset is gated)

`ARTPARK-IISc/Vaani-Noise-Event-Dataset` is gated, so every file request needs a token
from an account that has been granted access.

1. Open the [dataset page](https://huggingface.co/datasets/ARTPARK-IISc/Vaani-Noise-Event-Dataset)
   and click **Agree and access**.
2. Make a token at [hf.co/settings/tokens](https://huggingface.co/settings/tokens) (read scope).
3. In Colab: **Secrets** (key icon, left sidebar) -> add `HF_TOKEN` -> enable **Notebook access**.

The scripts find that secret on their own. The cell below just confirms it works.


In [ ]:
from huggingface_hub import HfApi
import sys; sys.path.insert(0, '/content/vaani-sed-track1')
from scripts.download_data import resolve_token, REPO

tok = resolve_token()
assert tok, 'No HF token found. Add HF_TOKEN under Colab Secrets and enable notebook access.'

who = HfApi(token=tok).whoami()
print('logged in as:', who.get('name'))
try:
    HfApi(token=tok).repo_info(REPO, repo_type='dataset', files_metadata=False)
    # a real file request is what gating actually blocks
    from huggingface_hub import hf_hub_download
    hf_hub_download(REPO, 'README.md', repo_type='dataset', token=tok)
    print('access to', REPO, 'CONFIRMED')
except Exception as e:
    print('NO ACCESS yet:', e)
    print("-> click 'Agree and access' on the dataset page, then re-run this cell")


## 3 · Download the dataset

**182 shards, 16.5 GB of parquet, 90,637 clips (~154.6 h).** Decoded to FLAC that is
roughly **9 GB**. It downloads straight to the local `$DATA` path (fast, and safe from
Drive's per-file overhead), then the sync cell after the full download copies it up to
`$DATA_DRIVE` so the next session finds it cached and skips the download entirely - the
mount cell above already pulled back whatever was synced last time. (Needs ~9 GB free
on Drive.)

Shards are fetched one at a time and each parquet blob is deleted once its clips are
written, so peak disk stays near the decoded size rather than decoded + 16.5 GB.
Everything resumes — after a disconnect, re-run and it continues.


See what is on the server (downloads nothing):


In [ ]:
!python scripts/download_data.py --out $DATA --list-only


**Start small.** Two shards is ~1000 clips — enough to confirm the whole pipeline end
to end in a few minutes before committing to the full download.


In [ ]:
!python scripts/download_data.py --out $DATA --max-shards 2

import json
print(json.dumps(json.load(open(f'{DATA}/stats.json')), indent=2)[:2500])


Then the full corpus. This is the long one — expect a while for 16.5 GB plus decode.
It skips whatever the previous cell already wrote.


In [ ]:
!python scripts/download_data.py --out $DATA


### Persist the dataset to Drive

Run this once the download above finishes (or any time you added more shards) so the
next session's mount cell finds it cached and skips the download entirely.


In [ ]:
if USE_DRIVE:
    import subprocess, time
    print('[data] syncing dataset to Drive for reuse next session ...')
    t0 = time.time()
    subprocess.run(['rsync', '-a', '--info=progress2', f'{DATA}/', f'{DATA_DRIVE}/'], check=True)
    print('[data] synced to %s in %.1fs' % (DATA_DRIVE, time.time() - t0))
else:
    print("[data] Drive not mounted - the dataset only lives on this session's ephemeral disk.")


### Check the tier split

The full corpus has an **`annotationQuality`** column, which is the gold/silver/bronze
signal the sample dataset lacked — so tiers are read from the data rather than assumed.

The download prints every value it saw. If any value could not be mapped it says so
loudly: add it to `QUALITY_ALIASES` in `src/data/prepare.py` and re-run (already
materialised clips are skipped, so it only rewrites the manifest and is quick).

A clip labelled gold/silver but carrying no timestamps is demoted to bronze — without
timestamps there is nothing for the frame-level loss to consume.


In [ ]:
import json, collections
recs = [json.loads(l) for l in open(f'{DATA}/manifest.jsonl', encoding='utf-8')]
tier_h = collections.Counter()
for r in recs: tier_h[r['tier']] += r['duration']
print('clips per tier :', dict(collections.Counter(r['tier'] for r in recs)))
print('hours per tier :', {k: round(v/3600, 2) for k, v in tier_h.items()})
print('states         :', len({r['state'] for r in recs}),
      '| languages:', len({r['language'] for r in recs}))
st = json.load(open(f'{DATA}/stats.json'))
print('annotationQuality values seen:', st.get('annotationQuality_values_seen'))
print('UNMAPPED (fix these!)        :', st.get('unmapped_annotationQuality'))


### Already downloaded before this fix? Repair tiers without re-downloading

A bug in `quality_to_tier` used to collapse every `unverified_timestamps` clip into
**gold** instead of **silver** (a substring-matching order bug: `"verified"` is
itself contained in `"unverified"`). If your manifest was built before this fix and
the tier check above shows **no silver clips at all**, run the cell below once - it
patches `tier` for the affected clips **in place** using only the (small)
annotationQuality column, re-fetched from each parquet shard. It does **not**
re-download or re-decode any audio, so it is fast even for the full corpus.

Safe to run even if your manifest is already correct - it just reports "nothing to
change" and exits.


In [ ]:
!python scripts/download_data.py --out $DATA --repair-tiers


## 4 · BEATs checkpoint (~360 MB, cached to Drive if mounted)


In [ ]:
from src.models.beats_encoder import download_beats
BEATS_DIR = f'{BASE}/checkpoints'   # under Drive when mounted, so this ~360 MB download
                                     # also survives a session restart
p = download_beats(BEATS_DIR)
print('BEATs at:', p)


## 5 · Train

One model, all three tiers. Every batch mixes gold/silver/bronze so mean-teacher
consistency and per-tier MixStyle always have material to work with.

On a T4, start with `--batch-size 16` if you hit OOM.

`--data $DATA` points at the **local** copy, never Drive directly. Reading random
small files straight off Drive's FUSE mount is what causes the "nothing happens for
20 minutes" stall before the first epoch prints anything; training here always runs
against local disk, and the config below points BEATs at the Drive-cached checkpoint
so that does not re-download either.


In [ ]:
import yaml

run_cfg = yaml.safe_load(open('configs/default.yaml'))
run_cfg['model']['beats_dir'] = BEATS_DIR
run_cfg['model']['beats_ckpt'] = str(p) if p else ''
RUN_CFG = f'{BASE}/config_run.yaml'
yaml.safe_dump(run_cfg, open(RUN_CFG, 'w'))

!python -m src.train.train \
    --config $RUN_CFG \
    --data $DATA \
    --out $RUN \
    --epochs 30 \
    --batch-size 24


### Training curve


In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open(f'{RUN}/history.json'))
fig, ax = plt.subplots(figsize=(7,4))
for which in ('student','teacher'):
    xs = [r['epoch'] for r in h if r['which']==which]
    ys = [r['score'] for r in h if r['which']==which]
    if xs: ax.plot(xs, ys, marker='o', label=which)
ax.set_xlabel('epoch'); ax.set_ylabel('0.5*F1 + 0.5*Dice'); ax.legend(); ax.grid(alpha=.3)
plt.show()
print('best:', max((r['score'] for r in h), default=None))


## 6 · Tune the post-processor

Runs on cached validation scores — **no retraining**. This is the cheapest large win
available; it also prints the plain median-filter baseline so you can see the delta.


In [ ]:
!python scripts/tune_postproc.py --run $RUN --rounds 2


## 7 · Predict → `submission.zip`

Point `--audio-dir` at the released test audio. Output is class-agnostic
onset/offset pairs, which is what Track 1 is scored on.

The archive is what you upload to
[Codabench competition 17825](https://www.codabench.org/competitions/17825/):
a ZIP holding a single `predictions.jsonl` at its root, one JSON object per clip.

In [ ]:
TEST_AUDIO = '/content/test_audio'   # <-- point at the official test set

import os
if os.path.isdir(TEST_AUDIO) and os.listdir(TEST_AUDIO):
    !python -m src.infer.predict \
        --ckpt $RUN/best.pt \
        --audio-dir $TEST_AUDIO \
        --params $RUN/postproc_params.json \
        --out submission.zip
else:
    print('No test audio yet - running on the training manifest as a demo instead.')
    !python -m src.infer.predict \
        --ckpt $RUN/best.pt \
        --manifest $DATA/manifest.jsonl \
        --params $RUN/postproc_params.json \
        --out submission.zip

In [ ]:
# Validate the archive against the competition's stated format before uploading.
import json, zipfile

with zipfile.ZipFile('submission.zip') as z:
    assert z.namelist() == ['predictions.jsonl'], z.namelist()
    lines = z.read('predictions.jsonl').decode('utf-8').strip().split('\n')

seen, n_ev = set(), 0
for line in lines:
    rec = json.loads(line)
    assert set(rec) == {'clip_id', 'events'}, rec.keys()
    assert rec['clip_id'] not in seen, 'duplicate clip_id ' + rec['clip_id']
    seen.add(rec['clip_id'])
    for ev in rec['events']:
        assert set(ev) == {'onset', 'offset'}
        assert 0.0 <= ev['onset'] <= ev['offset'], ev
    n_ev += len(rec['events'])

print('OK:', len(seen), 'clips,', n_ev, 'events')
for line in lines[:3]:
    print(line[:160])

### Download the submission


In [ ]:
from google.colab import files
files.download('submission.zip')

---
## Where to go next

The build order in the README is designed so you are always shippable:

1. ✅ this notebook = baseline + cSEBBs
2. **Frequency MixStyle sweep** — `model.mixstyle_p` in the config (0.3 / 0.5 / 0.7)
3. **Self-training** — pseudo-label the bronze tier with this model, promote confident
   predictions to silver, retrain. See `README.md`.
4. **Seed ensembling** — train 3 seeds, average frame scores, re-tune cSEBBs on the
   ensemble (re-tuning after ensembling matters; the score distribution shifts).

**Ablations worth running** (each is one config flag):

| Flag | Tests |
|---|---|
| `--no-beats` | how much BEATs is actually worth on Vaani |
| `model.n_basis: 1` | FDY vs plain CRNN — check per class, it can hurt fans/engines |
| `loss.lambda_cons: 0` | value of mean-teacher |
| `--method median` in tuning | cSEBBs vs frame thresholding |
